### ==========================================================
### Project     : Customer Churn Analytics Platform
### Notebook    : `01_Data_Ingestion`
### Layer       : Bronze
### Author      : Shivek Singh
### Created On  : 03-07-2026
#
### Purpose:
### Read the raw customer churn dataset from Unity Catalog
### Volume, validate the ingestion, and save it as a Bronze
### Delta table.
### ==========================================================

In [ ]:
# Local Jupyter / GitHub compatibility (no-op on Databricks)
import os
import sys
from pathlib import Path

if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    _root = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "utils" / "databricks_compat.py").exists()
        ),
        None,
    )
    if _root and str(_root / "src") not in sys.path:
        sys.path.insert(0, str(_root / "src"))
    from utils.databricks_compat import inject_globals

    inject_globals(globals())


In [ ]:

from pyspark.sql import functions as F


# Data Ingestion

In [ ]:

RAW_DATA_PATH = "/Volumes/workspace/default/customer_churn_data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

BRONZE_TABLE = "workspace.default.customer_churn_bronze"

In [ ]:

# ==========================================================
# Add Project src Directory to Python Path
# ==========================================================

import sys

PROJECT_ROOT = "/Workspace/Users/thakurshivek777@gmail.com/Customer-Churn-Platform"
SRC_PATH = f"{PROJECT_ROOT}/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print("Project src added successfully!")

In [ ]:

from config.settings import PROJECT_NAME

print(PROJECT_NAME)


## Read Dataset

In [ ]:

customer_df = (
    spark.read
         .format("csv")
         .option("header", True)
         .option("inferSchema", True)
         .load(RAW_DATA_PATH)
)


## Validate Dataset

In [ ]:

print("=" * 50)
print("Customer Churn Dataset - Initial Validation")
print("=" * 50)

print(f"Total Rows    : {customer_df.count()}")
print(f"Total Columns : {len(customer_df.columns)}")

In [ ]:

customer_df.printSchema()

In [ ]:

display(customer_df.limit(10))

In [ ]:

total_rows = customer_df.count()
unique_rows = customer_df.dropDuplicates().count()

print(f"Total Rows      : {total_rows}")
print(f"Unique Rows     : {unique_rows}")
print(f"Duplicate Rows  : {total_rows - unique_rows}")

In [ ]:

from pyspark.sql.functions import col, count, when

missing_df = customer_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customer_df.columns
])

display(missing_df)

In [ ]:

customer_df.groupBy("Churn").count().show()

In [ ]:

customer_df.describe().show()

In [ ]:

display(customer_df.dtypes)


## Save Bronze Layer

In [ ]:

(
    customer_df.write
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

In [ ]:

spark.read.table(BRONZE_TABLE).show(5)

In [ ]:

bronze_df = spark.read.table(BRONZE_TABLE)

display(bronze_df.limit(10))

In [ ]:

print(f"Rows : {bronze_df.count()}")
print(f"Columns : {len(bronze_df.columns)}")

In [ ]:

print("=" * 70)
print("BRONZE LAYER CREATED SUCCESSFULLY")
print("=" * 70)

print(f"Source File    : {RAW_DATA_PATH}")
print(f"Bronze Table   : {BRONZE_TABLE}")
print(f"Rows Loaded    : {bronze_df.count()}")
print(f"Columns Loaded : {len(bronze_df.columns)}")
print("Status         : SUCCESS")

In [ ]:

bronze_df.count()

In [ ]:

row_count = bronze_df.count()
column_count = len(bronze_df.columns)

In [ ]:

print(f"Rows Loaded    : {row_count}")
print(f"Columns Loaded : {column_count}")